# Análisis Gitleaks — Secretos en el Historial de Commits

Gitleaks escanea el historial completo de Git en busca de secretos filtrados:
API keys, tokens, contraseñas, credenciales de cloud, claves privadas, etc.
Los secretos expuestos en repos (incluso borrados luego) representan riesgo real.

**Fuente de datos:** colecciones `gitleaks_findings` y `gitleaks_scans` en `secpipeline.json`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DB_PATH = os.getenv("DB_PATH", "/data/secpipeline.json")
if not Path(DB_PATH).exists():
    DB_PATH = str(Path("../data/secpipeline.json").resolve())

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

with open(DB_PATH) as f:
    db = json.load(f)

repos  = pd.DataFrame(db.get("repositories", []))
scans  = pd.DataFrame(db.get("gitleaks_scans", []))
df     = pd.DataFrame(db.get("gitleaks_findings", []))

repo_names = repos.set_index("id")["full_name"].to_dict() if not repos.empty else {}

print(f"Repositorios escaneados con Gitleaks : {len(scans)}")
print(f"Total de secretos detectados         : {len(df)}")

if df.empty:
    print("\nNo hay hallazgos Gitleaks todavía. Ejecutá el miner primero.")
else:
    df["repo_name"] = df["repo_id"].map(repo_names)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    repos_with_secrets = df["repo_id"].nunique()
    repos_clean = len(scans) - repos_with_secrets
    print(f"Repositorios con secretos            : {repos_with_secrets}")
    print(f"Repositorios limpios                 : {repos_clean}")

## 1. Tipos de Secretos Detectados

Cada hallazgo corresponde a una regla de Gitleaks (e.g., `aws-access-token`, `github-pat`,
`generic-api-key`, `private-key`). Muestra qué tipo de credenciales están más expuestas.

In [ ]:
if not df.empty and "rule_id" in df.columns:
    rule_counts = df["rule_id"].value_counts().head(20)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    colors = sns.color_palette("Reds_r", len(rule_counts))
    axes[0].barh(rule_counts.index[::-1], rule_counts.values[::-1], color=colors[::-1])
    axes[0].set_xlabel("Cantidad de hallazgos")
    axes[0].set_title("Top 20 tipos de secretos")
    for i, v in enumerate(rule_counts.values[::-1]):
        axes[0].text(v + 0.2, i, str(v), va="center", fontsize=9)

    top_rules = rule_counts.head(8)
    other = rule_counts[8:].sum()
    if other > 0:
        top_rules = pd.concat([top_rules, pd.Series({"otros": other})])
    axes[1].pie(top_rules, labels=top_rules.index,
                autopct="%1.1f%%", startangle=90,
                colors=sns.color_palette("tab10", len(top_rules)))
    axes[1].set_title("Proporción de tipos de secretos")

    plt.suptitle("Clasificación de secretos detectados", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 2. Repositorios con Más Secretos Filtrados

Los repositorios con mayor cantidad de secretos requieren atención prioritaria.

In [ ]:
if not df.empty:
    repo_counts = df["repo_name"].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(11, max(5, len(repo_counts) * 0.4)))
    ax.barh(repo_counts.index[::-1], repo_counts.values[::-1], color="#d62728")
    ax.set_xlabel("Cantidad de secretos")
    ax.set_title("Repositorios con más secretos detectados", fontsize=13, fontweight="bold")
    for i, v in enumerate(repo_counts.values[::-1]):
        ax.text(v + 0.2, i, str(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

    # Repos sin secretos
    repos_scanned = set(scans["repo_id"].tolist()) if not scans.empty else set()
    repos_with_hits = set(df["repo_id"].tolist())
    clean_ids = repos_scanned - repos_with_hits
    print(f"\nRepositorios limpios ({len(clean_ids)}):")
    for rid in sorted(clean_ids):
        print(f"  ✓ {repo_names.get(rid, f'repo_{rid}')}")

## 3. Archivos con Más Secretos

Identifica archivos que concentran múltiples credenciales expuestas
(e.g., archivos `.env`, `config.py`, archivos de fixtures de tests).

In [ ]:
if not df.empty and "file_path" in df.columns:
    file_counts = df["file_path"].dropna().value_counts().head(20)

    if not file_counts.empty:
        labels = [p if len(p) <= 55 else "..." + p[-52:] for p in file_counts.index]

        fig, ax = plt.subplots(figsize=(12, 7))
        ax.barh(range(len(file_counts)), file_counts.values, color="#ff7f0e")
        ax.set_yticks(range(len(file_counts)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel("Cantidad de secretos")
        ax.set_title("Top 20 archivos con más secretos expuestos", fontsize=13, fontweight="bold")
        for i, v in enumerate(file_counts.values):
            ax.text(v + 0.2, i, str(v), va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

    # Extensiones más comunes
    df["extension"] = df["file_path"].dropna().apply(
        lambda p: Path(p).suffix.lower() if Path(p).suffix else "(sin ext)"
    )
    ext_counts = df["extension"].value_counts().head(15)
    print("\nExtensiones de archivos con secretos:")
    print(ext_counts.to_string())

## 4. Autores con Más Commits que Introdujeron Secretos

Permite identificar patrones de comportamiento o cuentas comprometidas.
Los emails se muestran sin anonimizar; considera la privacidad en informes externos.

In [ ]:
if not df.empty and "author" in df.columns:
    author_counts = df["author"].dropna().value_counts().head(15)

    if not author_counts.empty:
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.barh(author_counts.index[::-1], author_counts.values[::-1], color="#8c564b")
        ax.set_xlabel("Hallazgos")
        ax.set_title("Autores con más secretos filtrados", fontsize=13, fontweight="bold")
        for i, v in enumerate(author_counts.values[::-1]):
            ax.text(v + 0.1, i, str(v), va="center", fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print("No hay información de autores disponible.")

## 5. Línea de Tiempo de Commits con Secretos

Muestra cuándo se introdujeron secretos en el historial de cada repositorio.
Picos pueden indicar eventos específicos (onboarding de desarrolladores, migraciones, etc.).

In [ ]:
if not df.empty and "date" in df.columns:
    dated = df[df["date"].notna()].copy()

    if not dated.empty:
        dated["month"] = dated["date"].dt.to_period("M")
        monthly = dated.groupby("month").size().sort_index()

        fig, ax = plt.subplots(figsize=(14, 5))
        monthly.plot(kind="bar", ax=ax, color="#d62728")
        ax.set_xlabel("Mes")
        ax.set_ylabel("Secretos detectados")
        ax.set_title("Secretos por mes de commit", fontsize=13, fontweight="bold")
        tick_step = max(1, len(monthly) // 12)
        ax.set_xticks(range(0, len(monthly), tick_step))
        ax.set_xticklabels(
            [str(monthly.index[i]) for i in range(0, len(monthly), tick_step)],
            rotation=45, ha="right"
        )
        plt.tight_layout()
        plt.show()
    else:
        print("No hay información de fecha disponible en los hallazgos.")

## 6. Mapa Tipo de Secreto × Repositorio

Heatmap que cruza tipos de secretos con repositorios, revelando
si ciertos repos son afectados por tipos específicos de credenciales.

In [ ]:
if not df.empty and "rule_id" in df.columns:
    top_repos  = df["repo_name"].value_counts().head(15).index
    top_rules  = df["rule_id"].value_counts().head(10).index

    heat_data = (
        df[df["repo_name"].isin(top_repos) & df["rule_id"].isin(top_rules)]
        .groupby(["repo_name", "rule_id"])
        .size()
        .unstack(fill_value=0)
    )

    if not heat_data.empty:
        fig, ax = plt.subplots(figsize=(14, max(5, len(heat_data) * 0.5)))
        sns.heatmap(
            heat_data,
            annot=True,
            fmt="d",
            cmap="Reds",
            ax=ax,
            linewidths=0.5,
            cbar_kws={"label": "Hallazgos"},
        )
        ax.set_title("Tipo de secreto × Repositorio", fontsize=13, fontweight="bold")
        ax.set_ylabel("Repositorio")
        ax.set_xlabel("Tipo de secreto (rule_id)")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Datos insuficientes para el heatmap.")